## 🎯 Learning Objectives
* Understand the limitations of traditional fixed-size chunking for complex RAG queries.
* Grasp the concept of contextual retrieval, where larger chunks are initially retrieved to preserve context.
* Learn about late chunking (or 'post-retrieval re-chunking') as a technique to refine retrieved context for LLMs.
* Implement contextual retrieval and late chunking using LlamaIndex's `SentenceWindowNodeParser` and `ContextualCompressionRetriever`.
* Analyze the performance trade-offs and identify suitable use cases for these advanced RAG patterns.


## Contextual Retrieval and Late Chunking: Enhancing RAG Precision

In the evolving landscape of Retrieval Augmented Generation (RAG), simply retrieving fixed-size chunks based on semantic similarity often falls short. Imagine you're trying to understand a complex legal document. A single sentence might be semantically similar to your query, but without its surrounding paragraphs, its true meaning could be lost or misinterpreted. This is where **Contextual Retrieval** and **Late Chunking** come into play, offering a sophisticated approach to ensure the LLM receives not just relevant information, but *relevant information with sufficient context*.

### The Problem with Traditional Chunking

Traditional RAG often involves pre-chunking documents into fixed-size segments (e.g., 512 tokens). While simple, this approach has drawbacks:

1.  **Context Loss**: A critical piece of information might be split across two chunks, or a relevant sentence might lose its meaning without its immediate neighbors.
2.  **Noise**: Conversely, a chunk might contain a relevant sentence but also a lot of irrelevant information, diluting the signal for the LLM.
3.  **Suboptimal LLM Performance**: LLMs perform best when given precise, contextually rich information. Too little context leads to hallucination; too much irrelevant context can confuse the LLM or hit context window limits.

### Contextual Retrieval: Finding the Right Neighborhood

Contextual retrieval addresses the first problem by initially retrieving *larger, context-rich segments* of text. Instead of just a single sentence or a small paragraph, we aim to retrieve a 'window' of text that is likely to contain the answer and its necessary surrounding context. Think of it like this: instead of searching for a specific word on a page, you first identify the most relevant *chapter* or *section* of a book. This ensures that even if the exact answer is subtle, its broader context is available.

### Late Chunking: Zooming In for Precision

Once we have these larger, context-rich segments (our 'chapters'), we don't feed them directly to the LLM if they are still too large or contain too much noise. This is where **Late Chunking** (also known as post-retrieval re-chunking or contextual compression) becomes crucial. After retrieving the larger context, we then apply a secondary process to *extract the most relevant, smaller chunks* from within these larger segments. This is like having identified the relevant chapter, you then quickly scan it to find the exact paragraphs or sentences that directly answer your question.

**How it works in practice (LlamaIndex's approach):**

1.  **Initial Indexing**: Documents are parsed into smaller, overlapping 'sentence window' nodes. Each small node (e.g., a sentence) also stores a reference to a larger 'window' of text around it.
2.  **Retrieval**: When a query comes in, the retriever first identifies the most relevant *small nodes* (e.g., sentences) based on semantic similarity.
3.  **Contextual Expansion**: For each retrieved small node, its associated *larger window* of text is retrieved. This is the contextual retrieval step.
4.  **Compression/Re-ranking (Late Chunking)**: A `ContextualCompressionRetriever` then takes these larger windows and applies a 'compressor' (e.g., an LLM-based re-ranker or a sentence transformer re-ranker) to identify and extract only the most salient sentences or sub-segments from within those larger windows. This effectively 'chunks' the larger context down to the most relevant parts, just before sending it to the LLM.

This two-stage process ensures that the LLM receives highly relevant information, enriched with necessary context, but without being overwhelmed by irrelevant surrounding text. It's a powerful pattern for building robust, production-ready RAG systems in 2026 and beyond.


In [ ]:
# Ensure you have the necessary libraries installed
# pip install llama-index-llms-openai llama-index-embeddings-huggingface llama-index-readers-file llama-index-postprocessor-cohere_rerank

import os
from llama_index.core import Document, VectorStoreIndex, Settings
from llama_index.core.node_parser import SentenceWindowNodeParser
from llama_index.core.retrievers import AutoMergingRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.indices.postprocessor import MetadataReplacementPostProcessor, SentenceTransformerRerank
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# --- Configuration --- 
# Set your OpenAI API key. For production, use environment variables.
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

# For demonstration, we'll use a placeholder if the key isn't set
if "OPENAI_API_KEY" not in os.environ:
    print("Warning: OPENAI_API_KEY not found. Using a dummy key for demonstration. Please set it for actual use.")
    os.environ["OPENAI_API_KEY"] = "sk-dummykey"

# Configure LlamaIndex settings for 2026 best practices
# Using a robust embedding model and a capable LLM
Settings.llm = OpenAI(model="gpt-4o", temperature=0.1)
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-large-en-v1.5")
Settings.chunk_size = 512 # This is for the base document, not the sentence window

print("LlamaIndex Settings configured.")

# --- 1. Prepare Sample Document --- 
# Create a synthetic long document to demonstrate the concept
long_text = """
Agentic AI systems represent a paradigm shift in automation, moving beyond simple rule-based execution to intelligent, autonomous decision-making. These systems are designed to perceive their environment, reason about their goals, plan actions, and execute them, often with the ability to learn and adapt over time. The core components typically include a large language model (LLM) for reasoning, memory modules for retaining information, and tools for interacting with the external world.

One of the primary challenges in building robust agentic systems is managing their interaction with vast amounts of information. Retrieval Augmented Generation (RAG) has emerged as a critical technique to ground LLMs in up-to-date and domain-specific knowledge, mitigating hallucinations and improving factual accuracy. However, naive RAG implementations often struggle with complex queries that require synthesizing information from multiple, non-contiguous parts of a document.

Traditional RAG pipelines typically involve chunking documents into fixed-size segments and indexing them. When a query arrives, the most semantically similar chunks are retrieved and passed to the LLM. While effective for simple question-answering, this approach can lead to 'lost context' if a crucial piece of information is split across chunks, or 'noisy context' if a chunk contains much irrelevant data alongside the relevant part.

Advanced RAG patterns, such as contextual retrieval and late chunking, address these limitations. Contextual retrieval aims to fetch not just the most relevant small piece of information, but also its surrounding context. This is often achieved by indexing smaller units (like sentences) but associating them with larger 'windows' of text. When a small unit is retrieved, its larger window is also brought along.

Late chunking, or post-retrieval re-chunking, then refines this larger retrieved context. Instead of feeding the entire large window to the LLM, a 'compressor' or re-ranker is used to identify and extract only the most salient sentences or sub-segments from within that larger window. This ensures that the LLM receives a highly focused, yet contextually rich, input. For instance, a `SentenceWindowNodeParser` can create nodes representing individual sentences, each with a metadata field pointing to a larger surrounding text window. A `ContextualCompressionRetriever` then uses this structure.

Consider a scenario in financial analysis where an agent needs to understand the implications of a new regulatory change. The relevant information might be spread across several paragraphs, with key details in specific sentences. A simple RAG might retrieve only one sentence, missing the full picture. With contextual retrieval and late chunking, the system would first identify the relevant paragraphs, then precisely extract the critical sentences and their immediate context, providing the LLM with a comprehensive and accurate input for analysis.

This approach significantly enhances the precision and recall of RAG systems, leading to more reliable and insightful responses from agentic AI. It's particularly valuable for applications requiring deep understanding of complex documents, such as legal research, medical diagnostics, and technical documentation analysis.
"""

documents = [Document(text=long_text, id_="agentic_ai_rag_overview")]

# --- 2. Configure Sentence Window Node Parser --- 
# This parser creates nodes for individual sentences, but stores a larger 'window' of text
# around each sentence in its metadata. This is key for contextual retrieval.
node_parser = SentenceWindowNodeParser(
    window_size=3, # Number of sentences to include on each side of the central sentence
    sentence_splitter=lambda text: text.split(". "), # Simple splitter for demonstration
    overlap_ratio=0.5 # Overlap between windows
)

nodes = node_parser.get_nodes_from_documents(documents)

# --- 3. Build the Index --- 
# We build a vector index over these sentence window nodes.
# The embeddings are generated for the *central sentence* of each node.
vector_index = VectorStoreIndex(nodes)

print(f"Created index with {len(nodes)} nodes.")

# --- 4. Configure the Contextual Compression Retriever (Late Chunking) --- 
# This retriever first gets the top-k nodes, then expands them to their full window context,
# and finally uses a compressor to re-rank/filter the most relevant parts from those windows.

# Our compressor will be a SentenceTransformerRerank model for efficiency.
# For more complex scenarios, LLMRerank can be used.
rerank_model = SentenceTransformerRerank(
    model="cross-encoder/ms-marco-MiniLM-L-6-v2", 
    top_n=3 # Keep top 3 sentences after re-ranking within the expanded window
)

# The AutoMergingRetriever is a good choice for this pattern as it handles
# the expansion and compression logic. It uses the MetadataReplacementPostProcessor
# to replace the small node's text with its full window context before compression.
base_retriever = vector_index.as_retriever(similarity_top_k=5)

retriever = AutoMergingRetriever(
    base_retriever=base_retriever,
    node_parser=node_parser, # Pass the same node_parser used for indexing
    service_context=Settings # Use global settings for LLM/Embeddings
)

# The query engine combines the retriever with the LLM
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    node_postprocessors=[
        MetadataReplacementPostProcessor(target_metadata_key="window"), # Replace node text with its full window
        rerank_model # Apply the re-ranker (late chunking) to the expanded windows
    ]
)

print("Query engine configured with contextual retrieval and late chunking.")

# --- 5. Perform a Query and Observe Results --- 
query = "How do advanced RAG patterns like late chunking improve agentic AI?"

print(f"\nQuery: {query}")

# Get the raw retrieved nodes *before* compression for comparison
print("\n--- Raw Retrieved Nodes (before late chunking) ---")
raw_nodes = base_retriever.retrieve(query)
for i, node in enumerate(raw_nodes):
    print(f"Node {i+1} (Score: {node.score:.2f}):\n{node.text[:200]}...\n") # Show first 200 chars

# Get the final response and source nodes *after* contextual retrieval and late chunking
response = query_engine.query(query)

print("\n--- Final Response ---")
print(str(response))

print("\n--- Source Nodes (after late chunking) ---")
for i, node in enumerate(response.source_nodes):
    print(f"Source Node {i+1} (Score: {node.score:.2f}):\n{node.text}\n")

print("\nNotice how the 'Source Nodes' are more focused and precise, extracted from potentially larger initial contexts.")


### Interpreting the Code Output and Performance Trade-offs

The code demonstrates a powerful RAG pattern combining contextual retrieval and late chunking. Let's break down the output and discuss its implications:

1.  **Raw Retrieved Nodes (before late chunking)**: You'll observe that the `base_retriever` initially fetches nodes that are individual sentences (or small segments, depending on your `SentenceWindowNodeParser` configuration). These are the *central sentences* that are semantically most similar to your query. While relevant, they might lack the full surrounding context needed for a nuanced LLM response.

2.  **Source Nodes (after late chunking)**: The `query_engine`'s `response.source_nodes` will show the output *after* the `MetadataReplacementPostProcessor` has expanded the retrieved small nodes into their larger `window` contexts, and then the `SentenceTransformerRerank` model has compressed/re-ranked these larger windows to extract only the most relevant sentences. You should see that these final source nodes are more comprehensive than the raw retrieved sentences, yet more focused than the entire `window` would have been. They provide a richer, more precise context to the LLM.

### Performance Trade-offs and Use Cases

**Advantages:**

*   **Enhanced Relevance and Precision**: By first retrieving a broader context and then refining it, the LLM receives highly relevant information with sufficient surrounding details, leading to more accurate and comprehensive answers.
*   **Reduced Hallucination**: Grounding the LLM in precise, contextually rich information significantly mitigates the risk of generating factually incorrect or unsupported statements.
*   **Improved LLM Utilization**: The LLM's context window is used more efficiently, as it receives less irrelevant noise and more high-signal information.
*   **Better for Complex Queries**: Particularly effective for queries requiring synthesis of information, understanding nuances, or when the answer is spread across multiple, related sentences.

**Disadvantages:**

*   **Increased Latency**: The multi-stage retrieval and re-ranking process adds computational overhead, leading to higher query latency compared to simple RAG.
*   **Higher Computational Cost**: More embedding calls (for initial retrieval and potentially for re-ranking if using an LLM-based re-ranker) and potentially more complex processing. Using a `SentenceTransformerRerank` is generally faster than an `LLMRerank`.
*   **Complexity**: The setup is more involved than basic RAG, requiring careful configuration of node parsers, retrievers, and post-processors.

**Typical Use Cases (2026 and beyond):**

*   **Enterprise Knowledge Bases**: Answering complex questions from extensive internal documentation, legal contracts, or research papers where precise context is paramount.
*   **Medical and Scientific Research**: Summarizing findings, identifying relationships between concepts, or answering diagnostic questions from large bodies of literature.
*   **Financial Analysis**: Extracting specific details and their implications from financial reports, regulatory filings, or market analyses.
*   **Code Understanding and Generation**: Providing LLMs with highly relevant code snippets and their surrounding logical context for debugging, explanation, or generation tasks.
*   **Agentic AI Systems**: As demonstrated in the example, providing agents with highly refined and contextualized information to improve their planning, reasoning, and tool-use capabilities.

This pattern is a cornerstone for building truly intelligent and reliable RAG systems that can handle the complexities of real-world information.


### Resources

*   **LlamaIndex Documentation on Sentence Window Retrieval**: [https://docs.llamaindex.ai/en/stable/module_guides/querying/node_postprocessors/sentence_window_retrieval.html](https://docs.llamaindex.ai/en/stable/module_guides/querying/node_postprocessors/sentence_window_retrieval.html)
*   **LlamaIndex Documentation on Contextual Compression**: [https://docs.llamaindex.ai/en/stable/module_guides/querying/retriever/auto_merging_retriever.html](https://docs.llamaindex.ai/en/stable/module_guides/querying/retriever/auto_merging_retriever.html)
*   **LlamaIndex Postprocessors**: [https://docs.llamaindex.ai/en/stable/module_guides/querying/node_postprocessors/root.html](https://docs.llamaindex.ai/en/stable/module_guides/querying/node_postprocessors/root.html)
*   **Hugging Face Models (for Sentence Transformers)**: [https://huggingface.co/models](https://huggingface.co/models)
*   **OpenAI API Documentation**: [https://platform.openai.com/docs/](https://platform.openai.com/docs/)
*   **Research Paper on RAG (e.g., original RAG paper)**: [https://arxiv.org/abs/2005.11401](https://arxiv.org/abs/2005.11401) (While not directly on late chunking, it provides foundational RAG context.)
